# Wordle Solver: Comparative Evaluation

This notebook trains and evaluates five Wordle-solving strategies:

1. **Frequency Heuristic** — Letter frequency + positional scoring
2. **Information Gain (Minimax)** — Minimize worst-case remaining words
3. **DQN (Deep Q-Network)** — Neural network with curated exploration
4. **Tabular Q-Learning** — Learn which strategy to use per game state (Anderson & Meyer, 2022)
5. **Rollout** — One-step lookahead on info gain heuristic (Bhambri et al., 2022)

**Benchmark:** Bertsimas & Paskov (2024) optimal: 3.421 avg guesses, 100% win rate

### References
1. Bhambri, S. et al. (2022). *RL Methods for Wordle: A POMDP/Adaptive Control Approach.* arXiv:2211.10298.
2. Liu, C.-L. (2022). *Using Wordle for Learning to Design and Compare Strategies.* IEEE CoG.
3. Bertsimas, D. & Paskov, A. (2024). *An Exact and Interpretable Solution to Wordle.* Operations Research.
4. Anderson, B.J. & Meyer, J.G. (2022). *Finding the optimal human strategy for Wordle.* arXiv:2202.00557.
5. Ho, A. (2022). *Wordle Solving with Deep Reinforcement Learning.*

## Setup

In [ ]:
import os

GITHUB_TOKEN = ""  # your token
GITHUB_USER = "jhffmn82"
REPO = "wordle-ML-project"

if not os.path.exists(REPO):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO}.git

os.chdir(f"/content/{REPO}")

!git config user.email "jhoffmn.myAlt1@gmail.com"
!git config user.name "jhffmn82"
!git remote set-url origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO}.git

!pip install torch numpy matplotlib tqdm -q

In [ ]:
import sys, os, time, random, pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())

from engine.wordle_env import WordleGame, load_word_list, get_feedback, filter_words
from engine.state_encoder import encode_state, encode_words_onehot, STATE_DIM
from solvers.frequency_solver import FrequencySolver
from solvers.infogain_solver import InfoGainSolver
from solvers.dqn_solver import DQNNetwork, DQNSolver, ReplayBuffer
from solvers.tabular_q_solver import TabularQSolver, train_tabular_q, STRATEGY_NAMES
from solvers.rollout_solver import RolloutSolver

WORDS = load_word_list()
WORD_ENCODINGS = torch.tensor(encode_words_onehot(WORDS), dtype=torch.float32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loaded {len(WORDS)} words, device: {device}")

In [ ]:
def demo_solver(solver, words=None):
    """Play a few words and show colored results."""
    if words is None:
        words = ["crane", "slink", "nymph", "fuzzy", "vivid"]
    for target in words:
        print(f"\nTarget: {target.upper()}")
        game = WordleGame(target=target, word_list=WORDS)
        solver.reset()
        while not game.is_over():
            guess = solver.get_guess()
            feedback = game.make_guess(guess)
            solver.update(guess, feedback)
            tiles = " ".join(["🟩" if f==2 else "🟨" if f==1 else "⬛" for f in feedback])
            print(f"  Turn {game.turn}: {guess.upper()}  {tiles}")
            if game.is_solved():
                print(f"  Solved in {game.turn} guesses!")
                break
        if not game.is_solved():
            print(f"  Failed! Target was {target.upper()}")

---
## Part 1: Heuristic Solvers
These require no training.

### 1.1 Frequency Heuristic
Scores each word by letter frequency + 3x positional frequency among remaining candidates.

In [ ]:
freq_solver = FrequencySolver()
demo_solver(freq_solver)

### 1.2 Information Gain (Minimax)
For each candidate, partitions remaining words by feedback pattern. Picks the guess
that minimizes the largest partition (worst case).

In [ ]:
ig_solver = InfoGainSolver()
demo_solver(ig_solver)

---
## Part 2: Curated Word Lists

Run the frequency solver against every word and record which words are used at each turn.
Turn 1-2 guesses are pure information-gathering words — the best exploration vocabulary.

In [ ]:
from engine.word_lists import build_curated_lists

turn_words, set_a, set_b = build_curated_lists()

print(f"\nSet A (turns 1-2): {len(set_a)} words")
print(f"Set B (turns 1-3): {len(set_b)} words")
print(f"Full list: {len(WORDS)} words")

print(f"\nFirst 20 Set A words: {', '.join(w.upper() for w in set_a[:20])}")

---
## Part 3: DQN Training

Train a Deep Q-Network with curated exploration:
- Phase 1: exploration draws from Set A (narrowing words)
- Phase 2: exploration draws from Set B
- Phase 3: exploration from full list

Target word is always random from the full list.

In [ ]:
def compute_reward(feedback, solved, failed):
    reward = 0.0
    for fb in feedback:
        if fb == 2: reward += 5.0
        elif fb == 1: reward += 2.0
    if solved: reward += 25.0
    elif failed: reward -= 15.0
    return reward

In [ ]:
def train_dqn(num_episodes=50000, batch_size=64, gamma=0.95,
              epsilon_start=0.9, epsilon_end=0.05, epsilon_decay=0.99995,
              lr=0.001, target_update=500, log_interval=2000,
              exploration_sets=None):
    """
    Train DQN with curated exploration curriculum.
    
    exploration_sets: list of word lists for each phase.
        During exploration, random guesses are drawn from these sets.
        [set_a, set_b, WORDS] = 3 phases.
    """
    if exploration_sets is None:
        exploration_sets = [WORDS]
    
    word_to_idx = {w: i for i, w in enumerate(WORDS)}
    policy_net = DQNNetwork().to(device)
    target_net = DQNNetwork().to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    replay_buffer = ReplayBuffer(capacity=100000)
    word_enc = WORD_ENCODINGS.to(device)
    
    history = {"episode": [], "win_rate": [], "avg_guesses": [], "epsilon": []}
    recent_wins = []
    recent_guesses = []
    epsilon = epsilon_start
    
    episodes_per_phase = num_episodes // len(exploration_sets)
    
    for phase, explore_words in enumerate(exploration_sets):
        phase_start = phase * episodes_per_phase
        phase_end = (phase + 1) * episodes_per_phase if phase < len(exploration_sets) - 1 else num_episodes
        print(f"\n--- Phase {phase+1}: explore from {len(explore_words)} words "
              f"(episodes {phase_start}-{phase_end}) ---")
        
        for episode in range(phase_start, phase_end):
            target = random.choice(WORDS)
            game = WordleGame(target=target, word_list=WORDS)
            guesses, feedbacks_list = [], []
            remaining = list(WORDS)
            
            while not game.is_over():
                state = encode_state(guesses, feedbacks_list, game.turn)
                state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                
                if random.random() < epsilon:
                    # Curated exploration: pick from current phase's word set
                    # Prefer words still in remaining list
                    explore_remaining = [w for w in explore_words if w in set(remaining)]
                    if explore_remaining:
                        guess = random.choice(explore_remaining)
                    else:
                        guess = random.choice(explore_words)
                    action_idx = word_to_idx[guess]
                else:
                    with torch.no_grad():
                        output = policy_net(state_t)
                        scores = torch.matmul(output, word_enc.T).squeeze(0)
                        action_idx = scores.argmax().item()
                
                guess = WORDS[action_idx]
                feedback = game.make_guess(guess)
                guesses.append(guess)
                feedbacks_list.append(feedback)
                remaining = filter_words(remaining, guess, feedback)
                
                solved = game.is_solved()
                failed = game.turn >= 6 and not solved
                reward = compute_reward(feedback, solved, failed)
                next_state = encode_state(guesses, feedbacks_list, game.turn)
                done = solved or failed
                
                replay_buffer.push(state, action_idx, reward, next_state, done)
                
                if len(replay_buffer) >= batch_size:
                    s_b, a_b, r_b, ns_b, d_b = replay_buffer.sample(batch_size)
                    s_b, a_b, r_b = s_b.to(device), a_b.to(device), r_b.to(device)
                    ns_b, d_b = ns_b.to(device), d_b.to(device)
                    
                    out = policy_net(s_b)
                    q_vals = torch.matmul(out, word_enc.T).gather(1, a_b.unsqueeze(1)).squeeze(1)
                    
                    with torch.no_grad():
                        next_q = torch.matmul(target_net(ns_b), word_enc.T).max(dim=1)[0]
                        target_q = r_b + gamma * next_q * (1 - d_b)
                    
                    loss = F.mse_loss(q_vals, target_q)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                
                if done: break
            
            epsilon = max(epsilon_end, epsilon * epsilon_decay)
            if episode % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())
            
            recent_wins.append(1 if game.is_solved() else 0)
            recent_guesses.append(game.turn if game.is_solved() else 7)
            if len(recent_wins) > 1000: recent_wins.pop(0); recent_guesses.pop(0)
            
            if (episode + 1) % log_interval == 0:
                wr = sum(recent_wins)/len(recent_wins)
                ag = sum(recent_guesses)/len(recent_guesses)
                history["episode"].append(episode+1)
                history["win_rate"].append(wr)
                history["avg_guesses"].append(ag)
                history["epsilon"].append(epsilon)
                print(f"  Episode {episode+1}: win_rate={wr:.3f}, avg_guesses={ag:.2f}, eps={epsilon:.3f}")
    
    return policy_net, history

In [ ]:
print("Training DQN with curated exploration...")
start_time = time.time()

dqn_model, dqn_history = train_dqn(
    num_episodes=50000,
    exploration_sets=[set_a, set_b, WORDS],
    lr=0.001,
    log_interval=2000
)

print(f"\nDQN training: {(time.time()-start_time)/60:.1f} minutes")

os.makedirs("models", exist_ok=True)
torch.save(dqn_model.state_dict(), "models/dqn_model.pt")
print("Saved to models/dqn_model.pt")

In [ ]:
# DQN training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(dqn_history["episode"], dqn_history["win_rate"])
axes[0].set_title("DQN Win Rate"); axes[0].set_ylim(0, 1)
axes[0].axhline(y=1.0, color='r', linestyle='--', alpha=0.3)
axes[1].plot(dqn_history["episode"], dqn_history["avg_guesses"])
axes[1].set_title("DQN Avg Guesses")
axes[1].axhline(y=3.421, color='r', linestyle='--', alpha=0.3, label='Optimal')
axes[1].legend()
axes[2].plot(dqn_history["episode"], dqn_history["epsilon"])
axes[2].set_title("Epsilon")
plt.tight_layout(); plt.savefig("dqn_training.png", dpi=150); plt.show()

In [ ]:
dqn_solver = DQNSolver(model_path="models/dqn_model.pt")
demo_solver(dqn_solver)

---
## Part 4: Tabular Q-Learning (Anderson & Meyer, 2022)

The model doesn't pick words — it learns which **strategy** to use:
- State: (# greens, # yellows) from last guess (~30 states)
- Actions: 5 strategies (random, curated list, frequency, smart, exclude)
- Output: a policy table mapping each state to the best strategy

In [ ]:
print("Training Tabular Q-Learning...")
start_time = time.time()

q_table, q_history = train_tabular_q(
    num_episodes=10000,
    alpha=0.02,
    gamma=0.05,
    epsilon=0.3,
    curated_words=set_a,
    log_interval=1000
)

print(f"\nTabular Q training: {(time.time()-start_time)/60:.1f} minutes")

os.makedirs("models", exist_ok=True)
with open("models/q_table.pkl", "wb") as f:
    pickle.dump(q_table, f)
print("Saved to models/q_table.pkl")

In [ ]:
# Show learned policy
print("Learned policy:")
print(f"{'State':>12}  {'Best Strategy':>15}")
print("-" * 35)
for state in sorted(q_table.keys()):
    best = int(np.argmax(q_table[state]))
    print(f"  ({state[0]}g, {state[1]}y)  {STRATEGY_NAMES[best]:>15}")

In [ ]:
# Plot Q training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(q_history["episode"], q_history["win_rate"])
axes[0].set_title("Tabular Q Win Rate"); axes[0].set_ylim(0, 1)
axes[1].plot(q_history["episode"], q_history["avg_guesses"])
axes[1].set_title("Tabular Q Avg Guesses")
axes[1].axhline(y=3.421, color='r', linestyle='--', alpha=0.3, label='Optimal')
axes[1].legend()
plt.tight_layout(); plt.savefig("q_training.png", dpi=150); plt.show()

In [ ]:
tq_solver = TabularQSolver(q_table_path="models/q_table.pkl", curated_words=set_a)
demo_solver(tq_solver)

---
## Part 5: Rollout (Bhambri et al., 2022)

No training needed. At each turn:
1. Get top 5 candidates from info gain (minimax)
2. For each, simulate the full game forward for every remaining word
3. Pick the candidate with lowest average total guesses

Only activates when remaining words ≤ 20 (above that, minimax is used directly).

In [ ]:
rollout_solver = RolloutSolver(top_k=5, rollout_threshold=20)
demo_solver(rollout_solver)

---
## Part 6: Evaluation — All Five Solvers

Run each solver against every word. Measure win rate, average guesses, distribution.

In [ ]:
def evaluate_solver(solver, words, name="Solver", max_words=None):
    test_words = words[:max_words] if max_words else words
    results = []
    for target in tqdm(test_words, desc=name):
        game = WordleGame(target=target, word_list=words)
        solver.reset()
        while not game.is_over():
            guess = solver.get_guess()
            feedback = game.make_guess(guess)
            solver.update(guess, feedback)
            if game.is_solved(): break
        guesses_used = game.turn if game.is_solved() else 7
        results.append({"word": target, "guesses": guesses_used, "solved": game.is_solved()})
    
    guess_counts = [r["guesses"] for r in results]
    solved_counts = [r["guesses"] for r in results if r["solved"]]
    return {
        "name": name, "total_words": len(test_words),
        "solved": sum(1 for r in results if r["solved"]),
        "win_rate": sum(1 for r in results if r["solved"]) / len(test_words),
        "avg_guesses": np.mean(solved_counts) if solved_counts else float('inf'),
        "avg_guesses_all": np.mean(guess_counts),
        "max_guesses": max(guess_counts), "min_guesses": min(guess_counts),
        "distribution": guess_counts, "results": results
    }

def print_summary(s):
    print(f"\n{'='*50}")
    print(f"  {s['name']}")
    print(f"{'='*50}")
    print(f"  Words tested:    {s['total_words']}")
    print(f"  Solved:          {s['solved']} ({s['win_rate']*100:.1f}%)")
    print(f"  Avg guesses:     {s['avg_guesses']:.3f} (solved only)")
    print(f"  Avg guesses:     {s['avg_guesses_all']:.3f} (all, 7=fail)")
    print(f"  Min guesses:     {s['min_guesses']}")
    print(f"  Max guesses:     {s['max_guesses']}")
    dist = Counter(s['distribution'])
    print(f"\n  Distribution:")
    for k in sorted(dist.keys()):
        label = f"{k}" if k <= 6 else "7+ (fail)"
        bar = "█" * (dist[k] * 40 // s['total_words'])
        print(f"    {label:>10}: {dist[k]:>5} ({dist[k]/s['total_words']*100:5.1f}%) {bar}")

In [ ]:
MAX_WORDS = None  # Set to 50 for quick test, None for full eval
all_results = {}

In [ ]:
# Solver 1: Frequency Heuristic
freq_solver = FrequencySolver()
all_results["frequency"] = evaluate_solver(freq_solver, WORDS, "Frequency Heuristic", MAX_WORDS)
print_summary(all_results["frequency"])

In [ ]:
# Solver 2: Information Gain
ig_solver = InfoGainSolver()
all_results["infogain"] = evaluate_solver(ig_solver, WORDS, "Information Gain", MAX_WORDS)
print_summary(all_results["infogain"])

In [ ]:
# Solver 3: DQN
dqn_solver = DQNSolver(model_path="models/dqn_model.pt")
all_results["dqn"] = evaluate_solver(dqn_solver, WORDS, "DQN", MAX_WORDS)
print_summary(all_results["dqn"])

In [ ]:
# Solver 4: Tabular Q
tq_solver = TabularQSolver(q_table_path="models/q_table.pkl", curated_words=set_a)
all_results["tabular_q"] = evaluate_solver(tq_solver, WORDS, "Tabular Q-Learning", MAX_WORDS)
print_summary(all_results["tabular_q"])

In [ ]:
# Solver 5: Rollout
rollout_solver = RolloutSolver(top_k=5, rollout_threshold=20)
all_results["rollout"] = evaluate_solver(rollout_solver, WORDS, "Rollout", MAX_WORDS)
print_summary(all_results["rollout"])

---
## Part 7: Comparison & Visualization

In [ ]:
solver_keys = ["frequency", "infogain", "dqn", "tabular_q", "rollout"]

print(f"\n{'Solver':<25} {'Win Rate':>10} {'Avg Guesses':>12} {'Max':>5}")
print("-" * 55)
for key in solver_keys:
    r = all_results[key]
    print(f"{r['name']:<25} {r['win_rate']*100:>9.1f}% {r['avg_guesses']:>12.3f} {r['max_guesses']:>5}")
print("-" * 55)
print(f"{'Optimal (Bertsimas)':<25} {'100.0%':>10} {'3.421':>12} {'5':>5}")

In [ ]:
# Individual histograms
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Wordle Solver Comparison: Guess Distribution", fontsize=16, fontweight='bold')
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

for ax, key, color in zip(axes.flat[:5], solver_keys, colors):
    r = all_results[key]
    bins = range(1, 9)
    counts = [r["distribution"].count(i) for i in bins]
    labels = [str(i) if i <= 6 else "7+" for i in bins]
    ax.bar(labels, counts, color=color, edgecolor='white')
    for bar, c in zip(ax.patches, counts):
        if c > 0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(c), ha='center', fontsize=8)
    ax.set_title(f"{r['name']}\nAvg: {r['avg_guesses']:.3f} | Win: {r['win_rate']*100:.1f}%")
    ax.set_xlabel("Guesses"); ax.set_ylabel("Words")

axes.flat[5].axis('off')  # hide 6th subplot
plt.tight_layout(); plt.savefig("solver_comparison.png", dpi=150); plt.show()

In [ ]:
# Overlaid comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(1, 8)
width = 0.15
for i, (key, color) in enumerate(zip(solver_keys, colors)):
    r = all_results[key]
    counts = [r["distribution"].count(g) for g in range(1, 8)]
    offset = (i - 2) * width
    ax.bar(x + offset, counts, width, label=r["name"], color=color, edgecolor='white')

ax.set_xlabel("Number of Guesses", fontsize=12)
ax.set_ylabel("Number of Words", fontsize=12)
ax.set_title("Wordle Solver Comparison", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(i) if i <= 6 else "7+ (fail)" for i in range(1, 8)])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig("solver_overlay.png", dpi=150); plt.show()

In [ ]:
# Hardest words per solver
print("\nHardest words (most guesses) per solver:")
print("=" * 60)
for key in solver_keys:
    r = all_results[key]
    worst = sorted(r["results"], key=lambda x: -x["guesses"])
    print(f"\n{r['name']}:")
    for item in worst[:5]:
        s = "FAIL" if not item["solved"] else f"{item['guesses']} guesses"
        print(f"  {item['word'].upper():>8} — {s}")

---
## Summary

| Solver | Type | Win Rate | Avg Guesses | Reference |
|--------|------|----------|-------------|-----------|
| Frequency Heuristic | Heuristic | TBD | TBD | Baseline |
| Information Gain | Heuristic | TBD | TBD | Liu (2022) |
| DQN | Learned (RL) | TBD | TBD | Anderson & Meyer (2022), Ho (2022) |
| Tabular Q-Learning | Learned (RL) | TBD | TBD | Anderson & Meyer (2022) |
| Rollout | DP/Lookahead | TBD | TBD | Bhambri et al. (2022) |
| **Optimal** | **Exact DP** | **100%** | **3.421** | **Bertsimas & Paskov (2024)** |

In [ ]:
# === PUSH TO GITHUB ===
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null
!git add -A
!git commit -m "update from colab" --allow-empty
!git pull origin main --rebase
!git push